# What is Machine Learning?

An introduction to the core ideas behind machine learning: what it is, why it matters,
the main categories of ML algorithms, the standard workflow, and how to think about
model performance through the lens of overfitting and underfitting.

## 1. Defining Machine Learning

**Arthur Samuel (1959)** defined machine learning as:

> "The field of study that gives computers the ability to learn without being explicitly programmed."

A more precise, modern definition comes from **Tom Mitchell (1997)**:

> A computer program is said to *learn* from experience **E** with respect to some task **T**
> and some performance measure **P**, if its performance on **T**, as measured by **P**,
> improves with experience **E**.

In practical terms, machine learning is a set of techniques that allow computers to
discover patterns in data and use those patterns to make predictions or decisions,
rather than following a fixed set of hand-coded rules.

## 2. Why Machine Learning Matters

Traditional programming works well when the rules are **explicit and well-defined**.
Machine learning shines when:

| Scenario | Example |
|----------|---------|
| The rules are too complex to code by hand | Recognizing faces in photos |
| The rules change over time | Filtering spam email |
| The problem requires personalization | Product recommendations |
| Human expertise is hard to formalize | Medical diagnosis from imaging |
| The data is too large for manual analysis | Fraud detection across millions of transactions |

### Traditional Programming vs Machine Learning

```
Traditional Programming:
    Input Data + Rules  -->  Program  -->  Output

Machine Learning:
    Input Data + Expected Output  -->  Learning Algorithm  -->  Rules (Model)
```

In traditional programming, a human writes the rules. In machine learning, the
algorithm **discovers** the rules from data.

In [ ]:
# Traditional approach: hand-coded rules for classifying emails
def is_spam_traditional(email_text):
    """Rule-based spam filter -- brittle and hard to maintain."""
    spam_keywords = ["free money", "click here", "act now", "limited offer"]
    email_lower = email_text.lower()
    return any(keyword in email_lower for keyword in spam_keywords)


# With ML we would instead:
# 1. Collect thousands of labeled emails (spam / not spam)
# 2. Extract features (word frequencies, sender info, etc.)
# 3. Train a model that learns the boundary between spam and not-spam
# 4. The model generalizes to NEW emails it has never seen before

print(is_spam_traditional("Click here for FREE MONEY!"))   # True
print(is_spam_traditional("Meeting at 3pm tomorrow"))       # False
print(is_spam_traditional("You won a prize!"))              # False -- missed!

The rule-based filter missed `"You won a prize!"` because we did not anticipate that
pattern. A machine learning model trained on enough examples would likely catch it.

---
## 3. Types of Machine Learning

Machine learning algorithms are broadly grouped into three families based on the
**type of feedback** (supervision) available during training.

```
Machine Learning
├── Supervised Learning
│   ├── Classification
│   └── Regression
├── Unsupervised Learning
│   ├── Clustering
│   └── Dimensionality Reduction
└── Reinforcement Learning
```

### 3.1 Supervised Learning

The algorithm learns from **labeled** data -- each training example comes with the
correct answer (the *target* or *label*).

The goal is to learn a mapping function:

$$f: X \rightarrow y$$

where $X$ is the input (features) and $y$ is the output (label).

**Classification** -- the target is a **discrete category**.

| Task | Features | Target |
|------|----------|--------|
| Email spam detection | Word counts, sender | Spam / Not Spam |
| Image recognition | Pixel values | Cat / Dog / Bird |
| Disease diagnosis | Symptoms, lab results | Disease present / absent |

**Regression** -- the target is a **continuous number**.

| Task | Features | Target |
|------|----------|--------|
| House price prediction | Size, location, rooms | Price in dollars |
| Temperature forecasting | Historical weather data | Temperature in Celsius |
| Stock price prediction | Market indicators | Closing price |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Classification example ---
n = 50
class_0_x = np.random.randn(n) + 1
class_0_y = np.random.randn(n) + 1
class_1_x = np.random.randn(n) + 4
class_1_y = np.random.randn(n) + 4

axes[0].scatter(class_0_x, class_0_y, label="Class 0", alpha=0.7, edgecolors="k")
axes[0].scatter(class_1_x, class_1_y, label="Class 1", alpha=0.7, edgecolors="k")
axes[0].set_title("Classification (discrete target)")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")
axes[0].legend()

# --- Regression example ---
x_reg = np.linspace(0, 10, 60)
y_reg = 2.5 * x_reg + 5 + np.random.randn(60) * 3

axes[1].scatter(x_reg, y_reg, alpha=0.7, edgecolors="k")
axes[1].plot(x_reg, 2.5 * x_reg + 5, color="red", linewidth=2, label="True line")
axes[1].set_title("Regression (continuous target)")
axes[1].set_xlabel("Feature")
axes[1].set_ylabel("Target")
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.2 Unsupervised Learning

The algorithm works with **unlabeled** data -- there are no target values.
The goal is to discover hidden structure or patterns in the data.

**Clustering** -- group similar data points together.

- Customer segmentation (group customers by purchasing behavior)
- Document topic grouping
- Anomaly detection (unusual data points that do not fit any cluster)

**Dimensionality Reduction** -- compress data into fewer features while
preserving the most important information.

- PCA (Principal Component Analysis) for visualization
- Feature extraction before supervised learning
- Noise reduction in images

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# Generate synthetic clustered data (no labels given to the algorithm)
X_blobs, _ = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=42)

# KMeans discovers the clusters on its own
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], alpha=0.6, edgecolors="k")
axes[0].set_title("Raw data (no labels)")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, cmap="viridis",
                alpha=0.6, edgecolors="k")
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                marker="X", s=200, c="red", label="Centroids")
axes[1].set_title("After KMeans clustering")
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

# Iris dataset has 4 features -- reduce to 2 for visualization
iris = load_iris()
X_iris = iris.data     # shape (150, 4)
y_iris = iris.target   # used only for coloring, NOT given to PCA

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_iris)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=y_iris,
                      cmap="Set1", alpha=0.7, edgecolors="k")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("PCA: Iris dataset reduced from 4D to 2D")
plt.colorbar(scatter, label="Species")
plt.tight_layout()
plt.show()

print(f"Original shape: {X_iris.shape}")
print(f"Reduced shape:  {X_reduced.shape}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.1%}")

### 3.3 Reinforcement Learning

An **agent** learns by interacting with an **environment**. It takes actions,
receives rewards or penalties, and adjusts its strategy to maximize cumulative reward.

```
Agent  --action-->  Environment
Agent  <--state--   Environment
Agent  <--reward--  Environment
```

**Real-world examples:**

- Game playing (Chess, Go, Atari)
- Robotics (learning to walk, grasp objects)
- Autonomous driving (lane keeping, navigation)
- Resource management (data center cooling, ad bidding)

Reinforcement learning is conceptually different from supervised and unsupervised
learning and is typically studied as its own subfield. We will not cover it in depth
in this series, but it is important to know it exists.

### 3.4 Summary of ML Types

| Type | Data | Goal | Key Algorithms |
|------|------|------|----------------|
| **Supervised** | Labeled (X, y) | Predict y from X | KNN, Linear Regression, Decision Trees, SVM, Neural Networks |
| **Unsupervised** | Unlabeled (X only) | Discover structure | KMeans, DBSCAN, PCA, t-SNE |
| **Reinforcement** | Rewards from environment | Maximize cumulative reward | Q-Learning, Policy Gradient, DQN |

---
## 4. The Machine Learning Workflow

Every ML project follows a similar high-level pipeline. Understanding this workflow
is critical before diving into specific algorithms.

```
1. Define the Problem
       ↓
2. Collect & Explore Data
       ↓
3. Preprocess & Clean Data
       ↓
4. Split into Train / Test Sets
       ↓
5. Choose & Train a Model
       ↓
6. Evaluate the Model
       ↓
7. Tune & Improve
       ↓
8. Deploy
```

### Steps 1-2: Define the Problem and Explore Data

Before writing any code, clarify:
- What are you trying to predict or discover?
- Is this classification, regression, or clustering?
- What metric defines success?
- What data is available?

Then understand the data before modeling. Check shapes, types, distributions, missing
values, and correlations. This is **Exploratory Data Analysis (EDA)** -- the skills
from previous chapters (Pandas, Matplotlib, Seaborn) are essential here.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

# Load Iris as a DataFrame for exploration
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Shape: {df.shape}")
print(f"\nFeature types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
df.describe()

### Step 3: Preprocess and Clean Data

Common preprocessing tasks:
- Handle missing values (imputation or removal)
- Encode categorical variables (one-hot encoding, label encoding)
- Scale/normalize numerical features
- Remove duplicates and outliers

### Step 4: Train / Test Split

**Never evaluate a model on the same data it was trained on.** Split the data into:

- **Training set** (~70-80%): the model learns from this.
- **Test set** (~20-30%): held out to evaluate generalization.

This simulates how the model will perform on **unseen** data.

In [ ]:
from sklearn.model_selection import train_test_split

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

### Steps 5-6: Train and Evaluate

We will cover these in detail in the next notebooks. For now, here is a quick
preview using K-Nearest Neighbors.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Step 5: Train
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Step 6: Evaluate
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.2%}")

---
## 5. Overfitting vs Underfitting

One of the most important concepts in machine learning is understanding **how well
a model generalizes** to data it has never seen.

| Term | Description | Symptom |
|------|-------------|----------|
| **Underfitting** | Model is too simple to capture patterns | High error on both training and test data |
| **Good fit** | Model captures the true pattern | Low training error AND low test error |
| **Overfitting** | Model memorizes training data (including noise) | Very low training error BUT high test error |

### 5.1 Visual Intuition: Polynomial Fitting

A classic way to visualize overfitting is to fit polynomials of increasing degree
to noisy data generated from a simple function.

In [ ]:
from numpy.polynomial import polynomial as P

np.random.seed(0)

# True relationship: y = sin(x)
n_points = 20
x = np.sort(np.random.uniform(0, 2 * np.pi, n_points))
y_true = np.sin(x)
y_noisy = y_true + np.random.randn(n_points) * 0.3

x_smooth = np.linspace(0, 2 * np.pi, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ["Underfitting (degree=1)", "Good fit (degree=4)", "Overfitting (degree=15)"]
degrees = [1, 4, 15]

for ax, deg, title in zip(axes, degrees, titles):
    # Fit polynomial
    coeffs = np.polyfit(x, y_noisy, deg)
    poly_fn = np.poly1d(coeffs)

    ax.scatter(x, y_noisy, color="steelblue", edgecolors="k", zorder=5, label="Data")
    ax.plot(x_smooth, np.sin(x_smooth), "g--", linewidth=1.5, label="True function")
    ax.plot(x_smooth, poly_fn(x_smooth), "r-", linewidth=2, label=f"Degree {deg}")
    ax.set_ylim(-2, 2)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Observations:**

- **Degree 1** (left): A straight line cannot capture the sine wave -- it *underfits*.
- **Degree 4** (center): Captures the overall shape well -- a *good fit*.
- **Degree 15** (right): Passes through every noisy point but oscillates wildly
  between them -- it *overfits* the noise.

### 5.2 The Bias-Variance Tradeoff

The generalization error of a model can be decomposed into three components:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

| Component | Meaning |
|-----------|---------|
| **Bias** | Error from overly simplistic assumptions. High bias = underfitting. |
| **Variance** | Error from sensitivity to small fluctuations in training data. High variance = overfitting. |
| **Irreducible noise** | Inherent randomness in the data that no model can eliminate. |

The goal is to find the **sweet spot** where both bias and variance are reasonably low.

```
Error
  |\
  | \  Total Error
  |  \___________/
  |   \        /
  |    \      /  <-- Sweet spot
  | Bias \  / Variance
  |       \/
  +--------------------> Model Complexity
     Simple     Complex
```

In [ ]:
# Illustrate the bias-variance tradeoff conceptually
complexity = np.linspace(0.5, 10, 100)
bias_sq = 5 / complexity
variance = 0.05 * complexity ** 2
noise = np.full_like(complexity, 0.5)
total_error = bias_sq + variance + noise

plt.figure(figsize=(8, 5))
plt.plot(complexity, bias_sq, "b--", label="Bias$^2$", linewidth=2)
plt.plot(complexity, variance, "r--", label="Variance", linewidth=2)
plt.plot(complexity, noise, "g:", label="Irreducible noise", linewidth=2)
plt.plot(complexity, total_error, "k-", label="Total error", linewidth=2.5)

opt_idx = np.argmin(total_error)
plt.axvline(complexity[opt_idx], color="gray", linestyle=":", alpha=0.7)
plt.annotate("Optimal complexity", xy=(complexity[opt_idx], total_error[opt_idx]),
             xytext=(complexity[opt_idx] + 1.5, total_error[opt_idx] + 1),
             arrowprops=dict(arrowstyle="->"), fontsize=11)

plt.xlabel("Model Complexity", fontsize=12)
plt.ylabel("Error", fontsize=12)
plt.title("The Bias-Variance Tradeoff", fontsize=14)
plt.legend(fontsize=11)
plt.ylim(0, 8)
plt.tight_layout()
plt.show()

### 5.3 Detecting Overfitting in Practice

The most reliable signal is the **gap between training and test performance**.

Let us demonstrate this with KNN. As `k` decreases, the model becomes more complex
(at `k=1`, it memorizes the training set perfectly).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

k_values = range(1, 31)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train)))
    test_scores.append(accuracy_score(y_test, knn.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(k_values, train_scores, "o-", label="Training accuracy", markersize=4)
plt.plot(k_values, test_scores, "s-", label="Test accuracy", markersize=4)
plt.xlabel("k (number of neighbors)", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("KNN: Training vs Test Accuracy", fontsize=14)
plt.legend(fontsize=11)
plt.xticks(range(1, 31, 2))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Notice that at `k=1`, training accuracy is 100% (the model just looks up the nearest
training point -- itself). But test accuracy is lower. As `k` increases, training
accuracy drops (the model becomes simpler) while test accuracy first improves and
then may plateau or decline.

---
## 6. When to Use ML vs Traditional Programming

Machine learning is **not always the right tool**. Consider this decision guide:

**Use traditional programming when:**
- The rules are well-defined and unlikely to change
- The logic can be expressed as explicit if/else conditions
- Deterministic output is required (e.g., tax calculations)
- You do not have sufficient data to train a model

**Use machine learning when:**
- The rules are too complex or unknown
- The pattern changes over time
- You have enough labeled (or unlabeled) data
- A small error rate is acceptable
- Human-level performance requires intuition rather than rules

| Problem | Best Approach | Why |
|---------|---------------|-----|
| Sorting a list | Traditional | Well-defined algorithm |
| Calculating sales tax | Traditional | Explicit rules |
| Detecting fraudulent transactions | ML | Complex, evolving patterns |
| Recommending movies | ML | Personalization from user data |
| Converting Celsius to Fahrenheit | Traditional | Simple formula |

---
## 7. Real-World Examples by ML Type

### Supervised Learning -- Classification
- **Email spam filtering**: Gmail classifies incoming messages as spam or inbox
- **Medical imaging**: Detecting tumors in X-rays
- **Sentiment analysis**: Is a product review positive or negative?

### Supervised Learning -- Regression
- **Real estate**: Zillow's Zestimate predicts house prices
- **Weather forecasting**: Predicting tomorrow's temperature
- **Demand forecasting**: How many units will sell next quarter?

### Unsupervised Learning -- Clustering
- **Customer segmentation**: Group shoppers by behavior for targeted marketing
- **Genomics**: Grouping genes with similar expression patterns
- **Network security**: Detecting unusual traffic patterns

### Unsupervised Learning -- Dimensionality Reduction
- **Image compression**: Reducing pixel dimensions while preserving visual quality
- **Feature engineering**: Creating compact feature representations before modeling
- **Data visualization**: Projecting high-dimensional data to 2D for plotting

### Reinforcement Learning
- **AlphaGo / AlphaZero**: Learning to play board games at superhuman level
- **Robotics**: Teaching a robot arm to grasp objects
- **Resource optimization**: Google used RL to reduce data center cooling costs by 40%

---
## 8. Key Takeaways

1. **Machine learning** lets computers learn patterns from data instead of following
   hand-coded rules.

2. The three main types are **supervised** (labeled data), **unsupervised**
   (unlabeled data), and **reinforcement** (reward-based) learning.

3. The standard ML workflow goes from problem definition through data preparation,
   modeling, evaluation, and deployment.

4. **Overfitting** (memorizing noise) and **underfitting** (missing signal) are
   the central challenges -- the bias-variance tradeoff captures this tension.

5. ML is powerful but not universal -- use it when the problem involves complex
   or evolving patterns and sufficient data is available.

In the next notebook, we will get hands-on with **Scikit-Learn**, the most widely
used ML library in Python.